In [111]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch pandas numpy --quiet --break-system-packages

import os
import ast
import hashlib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [112]:
TARGET_FS = 500
TARGET_SECONDS = 10
TARGET_LEN = TARGET_FS * TARGET_SECONDS   # 5000
NUM_LEADS = 12
SEED = 42

In [113]:
DRIVE_ROOT = '/content/drive/MyDrive/LSTS/ecg_benchmark'
PROCESSED_DIR = f'{DRIVE_ROOT}/processed'

In [114]:
CLASS_NAMES = ['NSR', 'AFIB_AFL', 'IAVB', 'LBBB', 'RBBB']
NUM_CLASSES = len(CLASS_NAMES)

print("Setup complete. Looking for index CSVs in:", PROCESSED_DIR)

Setup complete. Looking for index CSVs in: /content/drive/MyDrive/LSTS/ecg_benchmark/processed


In [115]:
def parse_raw_labels(raw_labels_str, dataset):
    """
    Different datasets store raw_labels in different formats.
    Normalizes all of them into a plain list of code strings that
    can be looked up against mapping table.
    """
    if pd.isna(raw_labels_str) or raw_labels_str == '':
        return []

    if dataset == 'ptbxl':
        # PTB-XL's scp_codes column is a stringified dict, e.g.
        # "{'NORM': 100.0, 'LVH': 50.0}" — likelihood-weighted SCP codes.
        try:
            code_dict = ast.literal_eval(raw_labels_str)
            return list(code_dict.keys())
        except (ValueError, SyntaxError):
            return []

    elif dataset in ('cpsc2018', 'georgia', 'mimic_iv'):
        # Comma-separated SNOMED-CT (or SNOMED-mapped) codes,
        # e.g. "426783006,164889003"
        return [c.strip() for c in str(raw_labels_str).split(',') if c.strip()]

    elif dataset == 'code_ii':
        # CODE-II's own short codes, e.g. "1dAVb,RBBB"
        return [c.strip() for c in str(raw_labels_str).split(',') if c.strip()]

    else:
        raise ValueError(f"Unknown dataset for label parsing: {dataset}")


In [116]:
def build_label_vector(raw_labels_str, dataset, mapping_df, nsr_variant='A'):
    """
    Turns one recording's raw codes into a 5-element multi-hot vector,
    applying merge rules and the active NSR variant.

    Returns (label_vector, status):
      'ok'       — at least one code matched the mapping table
      'no_match' — none of this recording's codes are in the mapping
                   table at all (worth flagging — usually means an
                   unmapped code or a parsing mismatch)
    """
    codes = parse_raw_labels(raw_labels_str, dataset)
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    matched_any = False

    subset = mapping_df[mapping_df['dataset'] == dataset]

    for code in codes:
        rows = subset[subset['raw_code'] == code]
        if rows.empty:
            continue
        matched_any = True

        for _, row in rows.iterrows():
            if row['mapped_class'] == 'EXCLUDE':
                continue
            if row['mapped_class'] == 'NSR' and row['nsr_variant'] not in (nsr_variant, 'NA'):
                continue  # this row encodes the *other* NSR variant — skip
            class_idx = CLASS_NAMES.index(row['mapped_class'])
            vec[class_idx] = 1.0

    return vec, ('ok' if matched_any else 'no_match')


In [117]:
def assign_splits(index_df, seed=SEED, val_frac=0.1, test_frac=0.2):
    """
    Deterministic split based on hashing each recording's ID with the
    shared seed — same split every time, for anyone on the team,
    without needing to save a separate split file.

    NOTE: PTB-XL ships an official 10-fold split (its own strat_fold
    column), which the field treats as standard. This does NOT use
    that by default, to keep all 5 datasets on identical splitting
    logic. Flag to the team whether PTB-XL should be a special case
    before Week 2 locks in — switching later means rerunning every
    PTB-XL experiment.
    """
    def split_for(ecg_id):
        h = int(hashlib.md5(f"{seed}_{ecg_id}".encode()).hexdigest(), 16)
        frac = (h % 10000) / 10000.0
        if frac < test_frac:
            return 'test'
        elif frac < test_frac + val_frac:
            return 'val'
        else:
            return 'train'

    index_df = index_df.copy()
    index_df['split'] = index_df['ecg_id'].astype(str).apply(split_for)
    return index_df

In [118]:
class ECGDataset(Dataset):
    """
    The shared dataset class every training script imports.

    Usage:
        mapping_df = pd.read_csv(f'{PROCESSED_DIR}/mapping_table.csv')
        train_ds = ECGDataset(
            dataset_names=['ptbxl'],       # or ['ptbxl', 'cpsc2018', ...]
            split='train',
            index_dir=PROCESSED_DIR,
            mapping_df=mapping_df,
            nsr_variant='A',
        )
        x, y = train_ds[0]   # x: (12, 5000) tensor, y: (5,) multi-hot tensor
    """

    def __init__(self, dataset_names, split, index_dir, mapping_df,
                 nsr_variant='A', drop_no_match=True, verbose=True):
        assert split in ('train', 'val', 'test'), f"Unknown split: {split}"
        assert nsr_variant in ('A', 'B'), "nsr_variant must be 'A' or 'B'"

        self.nsr_variant = nsr_variant
        self.mapping_df = mapping_df
        self.index_dir = index_dir
        rows = []
        no_match_count = 0

        for dataset in dataset_names:
            index_path = f"{index_dir}/{dataset}_index.csv"
            if not os.path.exists(index_path):
                raise FileNotFoundError(
                    f"No index CSV found for '{dataset}' at {index_path}. "
                    f"Has that dataset's Week 1 preprocessing been run?"
                )

            df = pd.read_csv(index_path)

            if 'error' in df.columns:
                df = df[df['error'].isna()]
            if 'shape_ok' in df.columns:
                df = df[df['shape_ok'] == True]

            df = assign_splits(df)
            df = df[df['split'] == split]

            for _, row in df.iterrows():
                label_vec, status = build_label_vector(
                    row.get('raw_labels'), dataset, mapping_df, nsr_variant
                )
                if status == 'no_match':
                    no_match_count += 1
                    if drop_no_match:
                        continue

                rows.append({
                    'ecg_id': row['ecg_id'],
                    'dataset': dataset,
                    'label': label_vec,
                })

        self.samples = rows

        if verbose:
            print(f"[ECGDataset] {split} split, datasets={dataset_names}, "
                  f"nsr_variant={nsr_variant}: {len(self.samples)} recordings "
                  f"({no_match_count} unmatched-code recordings "
                  f"{'dropped' if drop_no_match else 'kept with zero label'})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        file_path = f"{self.index_dir}/{sample['dataset']}/signals/{sample['ecg_id']}.npy"
        sig = np.load(file_path)  # already (NUM_LEADS, TARGET_LEN)

        x = torch.from_numpy(sig).float()
        y = torch.from_numpy(sample['label']).float()
        return x, y

    def class_prevalence(self):
        """Per-class positive count for this split — use this to confirm
        the ~200-positives-per-fold requirement from Sydney's selection
        criteria still holds after splitting, not just before it."""
        labels = np.stack([s['label'] for s in self.samples])
        return {cls: int(labels[:, i].sum()) for i, cls in enumerate(CLASS_NAMES)}

In [119]:
def get_dataloader(dataset_names, split, index_dir, mapping_df,
                    batch_size=64, nsr_variant='A', num_workers=2,
                    shuffle=None):
    """Thin wrapper so training scripts don't each reimplement this.
    shuffle defaults to True for 'train', False otherwise."""
    ds = ECGDataset(dataset_names, split, index_dir, mapping_df, nsr_variant)
    if shuffle is None:
        shuffle = (split == 'train')
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=num_workers)

In [120]:
mock_mapping_rows = [
    # dataset,   raw_code,     mapped_class, rule_id,        nsr_variant
    ('ptbxl',    'SR',         'NSR',        'MOCK-A',       'A'),
    ('ptbxl',    'NORM',       'NSR',        'MOCK-B',       'B'),
    ('ptbxl',    'AFIB',       'AFIB_AFL',   'MOCK',         'NA'),
    ('ptbxl',    '1AVB',       'IAVB',       'MOCK',         'NA'),
    ('cpsc2018', 'Normal',     'NSR',        'MOCK-B',       'B'),
    ('cpsc2018', 'AF',         'AFIB_AFL',   'MOCK',         'NA'),
    ('cpsc2018', 'I-AVB',      'IAVB',       'MOCK',         'NA'),
]
mock_mapping_df = pd.DataFrame(
    mock_mapping_rows,
    columns=['dataset', 'raw_code', 'mapped_class', 'rule_id', 'nsr_variant']
)

# Only runs if your PTB-XL index CSV already exists from Week 1
if os.path.exists(f'{PROCESSED_DIR}/ptbxl_index.csv'):
    test_ds = ECGDataset(
        dataset_names=['ptbxl'],
        split='train',
        index_dir=PROCESSED_DIR,
        mapping_df=mock_mapping_df,
        nsr_variant='A',
        drop_no_match=False,   # keep everything so you can see the counts
    )
    print("Class prevalence in this split (mock mapping, small subset only):")
    print(test_ds.class_prevalence())

    x, y = test_ds[0]
    print("Sample signal shape:", x.shape, "| Sample label:", y)
else:
    print("ptbxl_index.csv not found — run your Week 1 prep notebook first.")

[ECGDataset] train split, datasets=['ptbxl'], nsr_variant=A: 15276 recordings (1473 unmatched-code recordings kept with zero label)
Class prevalence in this split (mock mapping, small subset only):
{'NSR': 11737, 'AFIB_AFL': 1056, 'IAVB': 580, 'LBBB': 0, 'RBBB': 0}
Sample signal shape: torch.Size([12, 5000]) | Sample label: tensor([1., 0., 0., 0., 0.])
